# Dataset loading and initial inspection
CPCB *Time Series Air Quality Data of India (2010-2023)*, stored in HDFS. Covers the loading checks (Task 7) and initial inspection (Task 8).

## 1. Environment check

In [ ]:
import sys, subprocess; sys.path.insert(0, '..')
from common import *
from pyspark.sql import functions as F
spark = get_spark('inspection')
print('Spark', spark.version, '| Hadoop', spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())
print('Raw data URI:', RAW_URI)

In [ ]:
print(subprocess.run(['bash', '-c', 'source ../hadoop/hadoop_env.sh && jps | sort -k2 && hdfs dfs -count -h /cpcb/raw && hdfs dfs -ls /cpcb'],
                     capture_output=True, text=True).stdout)

## 2. Load one file with header and schema inference

In [ ]:
one = spark.read.csv(f'{RAW_URI}/DL001.csv', header=True, inferSchema=True, sep=',')
one.printSchema()
print('rows:', one.count(), '| columns:', len(one.columns))

In [ ]:
one.show(5, truncate=False)

Header, delimiter, numeric types and both date columns (timestamp) are detected correctly for a single file.

## 3. Why all files cannot be loaded with one header

In [ ]:
import ingest
groups = ingest.scan_headers(spark)
print(len(groups), 'distinct header layouts across', sum(map(len, groups.values())), 'files')
for h in list(groups)[:3]:
    print(len(h), 'columns:', ', '.join(h[:7]), '...')

Spark applies one header positionally to every file, so the other 197 layouts would be misaligned. `ingest.py` reads each layout group with its own schema and harmonises them into one DataFrame (Bronze).

## 4. Initial inspection of the harmonised DataFrame

In [ ]:
df = spark.read.parquet(BRONZE_DIR)
print('rows:', f'{df.count():,}', '| columns:', len(df.columns))
print(df.columns)

In [ ]:
df.printSchema()

In [ ]:
df.orderBy('station_id', 'ts').show(10)

In [ ]:
df.orderBy(F.col('station_id').desc(), F.col('ts').desc()).show(5)

In [ ]:
df.select('pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'at', 'rh', 'ws').summary().show()

In [ ]:
stations = spark.read.parquet(STATIONS_DIR)
for c in ['state', 'city', 'agency']:
    print(c, stations.select(c).distinct().count())
stations.groupBy('state').count().orderBy(F.desc('count')).show(10)

In [ ]:
df.agg(F.min('ts'), F.max('ts'), F.min('pm25'), F.max('pm25'), F.min('co'), F.max('co')).show()

## 5. Data-quality checks

In [ ]:
nonempty = df.filter(F.coalesce(*[F.col(c) for c in MEASURES]).isNotNull())
print('empty placeholder rows:', df.count() - nonempty.count())
print('duplicate (station, hour):', df.count() - df.dropDuplicates(['station_id', 'ts']).count())
nonempty.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in MEASURES[:9]]).show()

In [ ]:
import pandas as pd
pd.read_csv(TABLES_DIR / 'data_quality_assessment.csv')

In [ ]:
spark.stop()